In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType
spark=SparkSession.builder.appName("SparkJoinDataFrame").getOrCreate()

data=[('01',"Santanu",1,1000),
      ('02',"Synthia",2,1100),
      ('03',"Suman",1,900),
      ('04',"Sunny",4,800),
      ('05',"Sunil",3,1000),
      ('06',"Surya",2,1200),
      ('07',"Suresh",2,1400),
      ('08',"Suvashish",2,500)]
columns =['id','name','deptid','salary']
df=spark.createDataFrame(data,columns)
df.show()
dept_columns=['deptid','deptname']
dept_data=[(1,'Data Science'),(2,"DataEngineer"),(4,"Data Analyst")]
schema= StructType([StructField("deptid",IntegerType(),True),StructField("Deptname",StringType(),True)])
dept_df=spark.createDataFrame(dept_data,schema)
dept_df.show()


+---+---------+------+------+
| id|     name|deptid|salary|
+---+---------+------+------+
| 01|  Santanu|     1|  1000|
| 02|  Synthia|     2|  1100|
| 03|    Suman|     1|   900|
| 04|    Sunny|     4|   800|
| 05|    Sunil|     3|  1000|
| 06|    Surya|     2|  1200|
| 07|   Suresh|     2|  1400|
| 08|Suvashish|     2|   500|
+---+---------+------+------+

+------+------------+
|deptid|    Deptname|
+------+------------+
|     1|Data Science|
|     2|DataEngineer|
|     4|Data Analyst|
+------+------------+



In [0]:
df.join(dept_df,df.deptid==dept_df.deptid).select(df["ID"].alias("Employee ID"),df["name"].alias("Employee Name"),dept_df["Deptname"].alias("Department")).show()

+-----------+-------------+------------+
|Employee ID|Employee Name|  Department|
+-----------+-------------+------------+
|         01|      Santanu|Data Science|
|         02|      Synthia|DataEngineer|
|         03|        Suman|Data Science|
|         04|        Sunny|Data Analyst|
|         06|        Surya|DataEngineer|
|         07|       Suresh|DataEngineer|
|         08|    Suvashish|DataEngineer|
+-----------+-------------+------------+



In [0]:
df.join(dept_df,df.deptid==dept_df.deptid,how='left').show()

+---+---------+------+------+------+------------+
| id|     name|deptid|salary|deptid|    Deptname|
+---+---------+------+------+------+------------+
| 01|  Santanu|     1|  1000|     1|Data Science|
| 02|  Synthia|     2|  1100|     2|DataEngineer|
| 03|    Suman|     1|   900|     1|Data Science|
| 04|    Sunny|     4|   800|     4|Data Analyst|
| 05|    Sunil|     3|  1000|  NULL|        NULL|
| 06|    Surya|     2|  1200|     2|DataEngineer|
| 07|   Suresh|     2|  1400|     2|DataEngineer|
| 08|Suvashish|     2|   500|     2|DataEngineer|
+---+---------+------+------+------+------------+



In [0]:
from pyspark.sql.functions import coalesce,lit

df.join(dept_df, df["deptid"] == dept_df["deptid"], how="left") \
  .select(
      df["id"].alias("Employee ID"),
      df["name"].alias("Employee Name"),
      coalesce(dept_df["deptname"],lit("NonDepartment")).alias("Department Name")
  ) \
  .show()

+-----------+-------------+---------------+
|Employee ID|Employee Name|Department Name|
+-----------+-------------+---------------+
|         01|      Santanu|   Data Science|
|         02|      Synthia|   DataEngineer|
|         03|        Suman|   Data Science|
|         04|        Sunny|   Data Analyst|
|         05|        Sunil|  NonDepartment|
|         06|        Surya|   DataEngineer|
|         07|       Suresh|   DataEngineer|
|         08|    Suvashish|   DataEngineer|
+-----------+-------------+---------------+



In [0]:
df.join(dept_df, df["deptid"] == dept_df["deptid"], how="leftanti") \
\
  .show()

+---+-----+------+------+
| id| name|deptid|salary|
+---+-----+------+------+
| 05|Sunil|     3|  1000|
+---+-----+------+------+



In [0]:
df_emp=df.join(dept_df, df["deptid"] == dept_df["deptid"], how="left") \
  .select(
      df["id"].alias("Employee ID"),
      df["name"].alias("Employee Name"),
      coalesce(dept_df["deptname"],lit("NonDepartment")).alias("Department Name"),
      df["salary"].alias("Emp salary")
  )
df_emp.show()

+-----------+-------------+---------------+----------+
|Employee ID|Employee Name|Department Name|Emp salary|
+-----------+-------------+---------------+----------+
|         01|      Santanu|   Data Science|      1000|
|         02|      Synthia|   DataEngineer|      1100|
|         03|        Suman|   Data Science|       900|
|         04|        Sunny|   Data Analyst|       800|
|         05|        Sunil|  NonDepartment|      1000|
|         06|        Surya|   DataEngineer|      1200|
|         07|       Suresh|   DataEngineer|      1400|
|         08|    Suvashish|   DataEngineer|       500|
+-----------+-------------+---------------+----------+



In [0]:
df_emp_grouped=df_emp.groupBy("Department Name").sum("Emp salary").alias("SumOfSalary")
df_emp_grouped.show()

+---------------+---------------+
|Department Name|sum(Emp salary)|
+---------------+---------------+
|   Data Science|           1900|
|   DataEngineer|           4200|
|   Data Analyst|            800|
|  NonDepartment|           1000|
+---------------+---------------+



In [0]:
from pyspark.sql import functions as F
df_emp_grade = df_emp.select(
    F.col("Employee ID").alias("Employee ID"),
    F.col("Employee Name").alias("Employee Name"),
    F.col("Department Name").alias("Department Name"),
    F.col("Emp salary").alias("Emp salary"),
    F.when(
        F.col("Emp salary") < 600, "Low Salary"
    ).when(
        (F.col("Emp salary") >= 600) & (F.col("Emp salary") < 900), "Medium Salary"
    ).when(
        (F.col("Emp salary") >= 900) & (F.col("Emp salary") < 1200), "High Salary"
    ).otherwise("Very High Salary").alias("Employee Salary Grade")
)
df_emp_grade.show()

+-----------+-------------+---------------+----------+---------------------+
|Employee ID|Employee Name|Department Name|Emp salary|Employee Salary Grade|
+-----------+-------------+---------------+----------+---------------------+
|         01|      Santanu|   Data Science|      1000|          High Salary|
|         02|      Synthia|   DataEngineer|      1100|          High Salary|
|         03|        Suman|   Data Science|       900|          High Salary|
|         04|        Sunny|   Data Analyst|       800|        Medium Salary|
|         05|        Sunil|  NonDepartment|      1000|          High Salary|
|         06|        Surya|   DataEngineer|      1200|     Very High Salary|
|         07|       Suresh|   DataEngineer|      1400|     Very High Salary|
|         08|    Suvashish|   DataEngineer|       500|           Low Salary|
+-----------+-------------+---------------+----------+---------------------+



In [0]:
df_emp_grade.groupBy("Department Name").pivot("Employee Salary Grade").avg("Emp salary").show()

+---------------+-----------+----------+-------------+----------------+
|Department Name|High Salary|Low Salary|Medium Salary|Very High Salary|
+---------------+-----------+----------+-------------+----------------+
|   Data Science|      950.0|      NULL|         NULL|            NULL|
|   DataEngineer|     1100.0|     500.0|         NULL|          1300.0|
|   Data Analyst|       NULL|      NULL|        800.0|            NULL|
|  NonDepartment|     1000.0|      NULL|         NULL|            NULL|
+---------------+-----------+----------+-------------+----------------+



User Defined Functions:-

In [0]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- deptid: long (nullable = true)
 |-- salary: long (nullable = true)



In [0]:
#old implementation:-
from pyspark.sql.functions import pandas_udf, PandasUDFType
@pandas_udf("id long, salary long", PandasUDFType.GROUPED_MAP)  # doctest: +SKIP
def normalize(pdf):
    salary = pdf.salary
    return pdf.assign(salary=(salary - salary.mean()) / salary.std())
df.groupby("id").apply(normalize).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/group.py:280: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


---------------------------------------------------------------------------
PythonException                           Traceback (most recent call last)
File <command-4615297105156886>, line 7
      5     salary = pdf.salary
      6     return pdf.assign(salary=(salary - salary.mean()) / salary.std())
----> 7 df.groupby("id").apply(normalize).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1123, in DataFrame.show(self, n, truncate, vertical)
   1122 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1123     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:876, in DataFrame._show_string(self, n, truncate, vertical)
    859     except ValueError:
    860         raise PySparkTypeError(
    861             errorClass="NOT_BOOL",
    862             messageParameters={
   (...)
    865             },
    866     